# 004 · Unicode, Bytes and Encodings

> **Standard Library › Text Processing** · Python 3.14

Python 3 draws a hard line between **text** (`str`, a sequence of Unicode code points) and **binary data** (`bytes` / `bytearray`). You convert between them explicitly with `encode` and `decode` — the same model as .NET's `string` vs `byte[]` with `Encoding`, but with a few important differences in how strings are measured and compared.

**In this notebook you will learn:**

- `str` vs `bytes` vs `bytearray`, and how they map to `string`, `byte[]` and `ReadOnlySpan<byte>`
- `encode` / `decode` and the `errors=` handlers (vs `Encoding.UTF8.GetBytes` / `GetString`)
- why `len("😀")` is 1 in Python but `"😀".Length` is 2 in C#
- `unicodedata`: character names, categories and normalisation (NFC / NFD)
- `casefold()` for caseless comparison, bytes literals, and hex conversion

## .NET mental model

| .NET | Python |
|---|---|
| `string` (UTF-16 code units) | `str` (Unicode **code points**) |
| `char` | no char type — a 1-character `str` |
| `Rune` (a code point) | a 1-character `str`; `ord()` / `chr()` |
| `byte[]` (mutable) | `bytearray` (mutable) |
| `ReadOnlySpan<byte>` / immutable bytes | `bytes` (immutable) |
| `Encoding.UTF8.GetBytes(s)` | `s.encode("utf-8")` |
| `Encoding.UTF8.GetString(b)` | `b.decode("utf-8")` |
| `EncoderFallback` / `DecoderFallback` | `errors="strict" \| "replace" \| "ignore" \| ...` |
| `s.Normalize(NormalizationForm.FormC)` | `unicodedata.normalize("NFC", s)` |
| `char.GetUnicodeCategory(c)` | `unicodedata.category(c)` |
| `Convert.ToHexString(b)` / `FromHexString` | `b.hex()` / `bytes.fromhex(s)` |
| `StringComparison.OrdinalIgnoreCase` | `a.casefold() == b.casefold()` |

## `str`, `bytes` and `bytearray`

- `str` — immutable text. Literals: `"..."`.
- `bytes` — immutable sequence of integers 0–255. Literals: `b"..."` (ASCII characters and `\x..` escapes only).
- `bytearray` — the **mutable** version of `bytes`; the closest thing to a C# `byte[]`.

In [ ]:
text = "héllo"
data = b"h\xc3\xa9llo"          # bytes literal: ASCII + \x escapes
buf = bytearray(b"abc")

print(type(text).__name__, type(data).__name__, type(buf).__name__)
print(data[0], list(b"hi"))     # indexing bytes gives an int (like C# byte)
print(data[:1])                  # slicing gives bytes

buf[0] = 65                      # bytearray is mutable
buf.extend(b"!")
print(buf, bytes(buf))

⚠️ **Gotcha:** `str` and `bytes` never mix implicitly. Concatenating or comparing them doesn't coerce — you get a `TypeError` (or simply `False` for `==`).

In [ ]:
try:
    "abc" + b"def"
except TypeError as e:
    print(f"{type(e).__name__}: {e}")

print("abc" == b"abc")     # False — different types, never equal

## Encoding and decoding

`str.encode(encoding)` produces `bytes`; `bytes.decode(encoding)` produces `str`. Both default to **UTF-8**.

```csharp
// C#
byte[] bytes = Encoding.UTF8.GetBytes("héllo");
string text  = Encoding.UTF8.GetString(bytes);
```

In [ ]:
encoded = "héllo".encode()            # same as .encode("utf-8")
print(encoded, len(encoded))          # é takes two bytes in UTF-8
print(encoded.decode("utf-8"))

print("héllo".encode("utf-16-le"))    # what a .NET string looks like in memory
print("héllo".encode("latin-1"))      # one byte per character

## Error handlers: `errors=`

By default both directions are **strict**: an unencodable character or an invalid byte sequence raises `UnicodeEncodeError` / `UnicodeDecodeError`.

⚠️ **Gotcha:** this is the opposite of .NET's default. `Encoding.UTF8.GetString` silently replaces invalid bytes with `U+FFFD` (�) unless you construct `new UTF8Encoding(false, throwOnInvalidBytes: true)`. Python fails loudly unless you opt in to leniency.

In [ ]:
raw = "héllo".encode("utf-8")

try:
    raw.decode("ascii")
except UnicodeDecodeError as e:
    print(f"{type(e).__name__}: {e}")

try:
    "price: 5€".encode("ascii")
except UnicodeEncodeError as e:
    print(f"{type(e).__name__}: {e}")

Pick a handler with `errors=`:

| Handler | Decoding | Encoding |
|---|---|---|
| `"strict"` (default) | raise | raise |
| `"replace"` | `�` (U+FFFD) | `?` |
| `"ignore"` | drop bytes | drop characters |
| `"backslashreplace"` | `\xc3` | `\u20ac` |
| `"xmlcharrefreplace"` | — | `&#8364;` |
| `"surrogateescape"` | smuggle bad bytes through as lone surrogates (round-trips) | restore them |

In [ ]:
print(raw.decode("ascii", errors="replace"))
print(raw.decode("ascii", errors="ignore"))
print(raw.decode("ascii", errors="backslashreplace"))

s = "price: 5€"
for handler in ["replace", "ignore", "backslashreplace", "xmlcharrefreplace"]:
    print(f"{handler:18}", s.encode("ascii", errors=handler))

**Tip:** UTF-8 is the default for `str.encode()` / `bytes.decode()`, but **not** necessarily for `open()` — text files use the platform's locale encoding unless you pass `encoding=` (on Windows that can still be a legacy code page). Always write `open(path, encoding="utf-8")` *(see Standard Library › Files & Paths › 002 · Reading and Writing Files)*.

## Code points vs UTF-16 code units

A C# `string` is a sequence of UTF-16 **code units**: characters outside the Basic Multilingual Plane (most emoji, many CJK extensions) take *two* `char`s — a surrogate pair. A Python `str` is a sequence of **code points**, so each such character counts as one.

```csharp
// C#
Console.WriteLine("😀".Length);                    // 2  (surrogate pair)
Console.WriteLine("😀".EnumerateRunes().Count());  // 1
```

In [ ]:
emoji = "😀"
print(len(emoji))                          # 1 — one code point
print(len(emoji.encode("utf-16-le")) // 2) # 2 — what C# .Length reports
print(len(emoji.encode("utf-8")))          # 4 bytes in UTF-8
print(hex(ord(emoji)), chr(0x1F600))       # ord/chr ≈ Rune.Value / new Rune(...)

So indexing and slicing in Python never split a surrogate pair — `s[0]` is always a full code point. Escapes come in three sizes plus a by-name form:

In [ ]:
print("\u20ac", "\U0001F600", "\N{EURO SIGN}", "\x41")
print(ord("€"), chr(8364))

⚠️ **Gotcha:** code points are still not "characters as a user sees them". A thumbs-up with a skin-tone modifier, a flag, or a family emoji are **several** code points forming one *grapheme cluster*. Python's standard library has no grapheme API (.NET has `StringInfo.LengthInTextElements` / `StringInfo.GetTextElementEnumerator`), so `len()` over-counts these — use the third-party `regex` module (`\X`) or a dedicated package if you need it.

In [ ]:
import unicodedata

thumbs = "👍🏽"
print(len(thumbs))
print([unicodedata.name(ch) for ch in thumbs])

## `unicodedata`: names and categories

`unicodedata` exposes the Unicode Character Database: names, general categories, numeric values and more.

```csharp
// C#
var cat = char.GetUnicodeCategory('é');   // UnicodeCategory.LowercaseLetter
```

In [ ]:
for ch in ["é", "A", "7", "€", "😀", " "]:
    print(repr(ch), unicodedata.name(ch), unicodedata.category(ch))

print(unicodedata.lookup("SNOWMAN"))      # name -> character
print(unicodedata.unidata_version)       # the Unicode version this Python ships

Categories are two-letter codes: `Lu`/`Ll` (upper/lowercase letter), `Nd` (decimal digit), `Sc` (currency symbol), `So` (other symbol), `Zs` (space separator), `Mn` (non-spacing mark — e.g. combining accents), and so on.

## Normalisation: NFC vs NFD

The same visible text can have different code-point sequences: `é` can be one precomposed code point (U+00E9) or `e` followed by a combining acute accent (U+0301). They look identical but compare **unequal** — in both Python and C# (with ordinal comparison).

```csharp
// C#
var nfc = s.Normalize(NormalizationForm.FormC);
bool same = a.Normalize() == b.Normalize();
```

In [ ]:
composed = "caf\u00e9"          # é as one code point
decomposed = "cafe\u0301"       # e + combining acute accent

print(composed, decomposed, composed == decomposed)
print(len(composed), len(decomposed))

nfc = unicodedata.normalize("NFC", decomposed)
print(nfc == composed, len(nfc))
print(unicodedata.is_normalized("NFC", composed))

| Form | Meaning | .NET |
|---|---|---|
| `NFC` | composed — the usual choice for storage and comparison | `NormalizationForm.FormC` |
| `NFD` | decomposed | `NormalizationForm.FormD` |
| `NFKC` / `NFKD` | also fold "compatibility" variants (ligatures, circled digits, full-width letters) | `FormKC` / `FormKD` |

A classic trick — strip accents by decomposing and dropping combining marks (category `Mn`):

In [ ]:
def strip_accents(s: str) -> str:
    return "".join(ch for ch in unicodedata.normalize("NFD", s)
                   if unicodedata.category(ch) != "Mn")

print(strip_accents("Crème brûlée à São Paulo"))
print(unicodedata.normalize("NFKC", "ﬁle ①"))   # ligature and circled digit folded

## Caseless comparison: `casefold()` vs `lower()`

`lower()` is fine for display, but for **comparisons** use `casefold()`: it applies the full Unicode case-folding rules, e.g. German `ß` folds to `ss`.

```csharp
// C#
bool eq = string.Equals(a, b, StringComparison.OrdinalIgnoreCase);
// OrdinalIgnoreCase uses simple per-char mapping: "straße" != "STRASSE"
```

In [ ]:
a, b = "Straße", "STRASSE"
print(a.lower(), b.lower(), a.lower() == b.lower())
print(a.casefold(), b.casefold(), a.casefold() == b.casefold())

**Tip:** for robust caseless matching of user input, normalise **and** case-fold: `unicodedata.normalize("NFC", s).casefold()` (the Unicode standard recommends NFD/NFKD around casefolding for full canonical caseless matching).

## Bytes literals and hex

`bytes.hex()` and `bytes.fromhex()` convert to and from hex strings — like `Convert.ToHexString` / `Convert.FromHexString` (but lowercase, and with an optional separator).

```csharp
// C#
string hex = Convert.ToHexString(bytes);   // "68C3A9"
byte[] back = Convert.FromHexString(hex);
```

In [ ]:
data = "hé".encode()
print(data.hex())               # lowercase, no separator
print(data.hex(" "))            # with a separator
print(data.hex().upper())       # to match .NET's uppercase output
print(bytes.fromhex("68 c3 a9"))  # whitespace is allowed
print(bytes([72, 105]), bytes(3))  # from ints; bytes(n) = n zero bytes

⚠️ **Gotcha:** `bytes(5)` creates **five zero bytes**, not `b"5"`. To turn a number into its text bytes use `str(5).encode()`, or `int.to_bytes` for the binary representation.

In [ ]:
print(bytes(5))
print(str(5).encode())
print((1025).to_bytes(2, "big"), int.from_bytes(b"\x04\x01", "big"))   # BinaryPrimitives-style

## Summary

| Task | C# / .NET | Python |
|---|---|---|
| Text → bytes | `Encoding.UTF8.GetBytes(s)` | `s.encode()` (UTF-8 default) |
| Bytes → text | `Encoding.UTF8.GetString(b)` | `b.decode()` |
| Invalid bytes | replaced with `�` by default | `UnicodeDecodeError` by default |
| Lenient decode | `DecoderFallback` | `b.decode(errors="replace")` |
| Mutable byte buffer | `byte[]` | `bytearray` |
| Length of `"😀"` | `.Length` → 2 | `len()` → 1 |
| Code point ↔ int | `Rune.Value` / `new Rune(n)` | `ord(c)` / `chr(n)` |
| Unicode category | `char.GetUnicodeCategory(c)` | `unicodedata.category(c)` |
| Character name | — | `unicodedata.name(c)` |
| Normalise | `s.Normalize(NormalizationForm.FormC)` | `unicodedata.normalize("NFC", s)` |
| Caseless compare | `StringComparison.OrdinalIgnoreCase` | `a.casefold() == b.casefold()` |
| Hex string | `Convert.ToHexString(b)` | `b.hex()` |
| Parse hex | `Convert.FromHexString(s)` | `bytes.fromhex(s)` |

## Exercises

1. Write `utf16_length(s)` that returns what C#'s `string.Length` would report for `s`. Check it with `"abc"`, `"é"` and `"😀👍"`.
2. Write `same_text(a, b)` that returns `True` when two strings are equal ignoring case **and** composed/decomposed differences. Test with `"CAFÉ"` and `"cafe\u0301"`.
3. Decode the bytes `b"caf\xe9 ok"` (Latin-1 data) as UTF-8 without raising, showing the invalid byte as a backslash escape. Then decode it correctly.
4. Produce a "slug" from `"Crème Brûlée Recipe!"` → `"creme-brulee-recipe"` using `unicodedata` and `re`.

In [ ]:
# Your solutions here

<details>
<summary>Show solutions</summary>

```python
import re
import unicodedata

# 1 — each UTF-16 code unit is 2 bytes
def utf16_length(s: str) -> int:
    return len(s.encode("utf-16-le")) // 2

print(utf16_length("abc"), utf16_length("é"), utf16_length("😀👍"))   # 3 1 4

# 2 — normalise, then casefold
def same_text(a: str, b: str) -> bool:
    norm = lambda s: unicodedata.normalize("NFC", s).casefold()
    return norm(a) == norm(b)

print(same_text("CAFÉ", "cafe\u0301"))

# 3
raw = b"caf\xe9 ok"
print(raw.decode("utf-8", errors="backslashreplace"))
print(raw.decode("latin-1"))

# 4 — strip accents, lowercase, collapse non-alphanumerics to "-"
def slugify(s: str) -> str:
    ascii_text = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode()
    return re.sub(r"[^a-z0-9]+", "-", ascii_text.lower()).strip("-")

print(slugify("Crème Brûlée Recipe!"))
```

</details>